# S02 — Feature Engineering QA

Visual validation of Phase 2 outputs: risk index construction, feature distributions, and data quality checks.

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA = ROOT / "data"
full = pd.read_parquet(DATA / "spain_grid_features.parquet")
train = pd.read_parquet(DATA / "train.parquet")
val = pd.read_parquet(DATA / "val.parquet")
test = pd.read_parquet(DATA / "test.parquet")
fit = joblib.load(DATA / "artifacts" / "risk_index_fit.joblib")

print(f"Full: {len(full):,}  Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}")

Full: 44,683  Train: 25,568  Val: 4,317  Test: 14,798


In [4]:
full.head(50)

,actual_demand_mw,forecast_demand_mw,gen_wind_mw,gen_solar_pv_mw,gen_hydro_mw,gen_combined_cycle_mw,gen_nuclear_mw,gen_total_mw,forecast_wind_mw,forecast_solar_mw,...,demand_forecast_error,net_load,risk_index,risk_category,hour_of_day,day_of_week,month,is_weekend,risk_index_lag_24h,risk_index_lag_168h
2021-01-07 23:00:00+00:00,31124.000000,30752.666667,226266.640,6751.867,147776.143,88427.870,171013.866,640236.386,106125.353,9459.079,...,371.333333,-201894.507000,0.531727,Medium,0,4,1,0,0.803966,0.405585
2021-01-08 00:00:00+00:00,28430.750000,28270.916667,226266.640,6751.867,147776.143,88427.870,171013.866,640236.386,106125.353,9459.079,...,159.833333,-204587.757000,0.528698,Medium,1,4,1,0,0.801225,0.404112
2021-01-08 01:00:00+00:00,26938.083333,26725.916667,226266.640,6751.867,147776.143,88427.870,171013.866,640236.386,106125.353,9459.079,...,212.166667,-206080.423667,0.526945,Medium,2,4,1,0,0.799747,0.401575
2021-01-08 02:00:00+00:00,26226.666667,26157.583333,226266.640,6751.867,147776.143,88427.870,171013.866,640236.386,106125.353,9459.079,...,69.083333,-206791.840333,0.526183,Medium,3,4,1,0,0.799073,0.399419
2021-01-08 03:00:00+00:00,25991.833333,25967.833333,226266.640,6751.867,147776.143,88427.870,171013.866,640236.386,106125.353,9459.079,...,24.000000,-207026.673667,0.525930,Medium,4,4,1,0,0.799018,0.398336
2021-01-08 04:00:00+00:00,26591.916667,26540.500000,226266.640,6751.867,147776.143,88427.870,171013.866,640236.386,106125.353,9459.079,...,51.416667,-206426.590333,0.526614,Medium,5,4,1,0,0.799912,0.398067
2021-01-08 05:00:00+00:00,29146.416667,28979.333333,226266.640,6751.867,147776.143,88427.870,171013.866,640236.386,106125.353,9459.079,...,167.083333,-203872.090333,0.529524,Medium,6,4,1,0,0.803304,0.398276
2021-01-08 06:00:00+00:00,33417.666667,34061.250000,226266.640,6751.867,147776.143,88427.870,171013.866,640236.386,106125.353,9459.079,...,-643.583333,-199600.840333,0.534825,Medium,7,4,1,0,0.808175,0.398778
2021-01-08 07:00:00+00:00,37093.916667,37300.416667,226266.640,6751.867,147776.143,88427.870,171013.866,640236.386,106125.353,9459.079,...,-206.500000,-195924.590333,0.538895,Medium,8,4,1,0,0.812680,0.399123
2021-01-08 08:00:00+00:00,40056.583333,40084.333333,226266.640,6751.867,147776.143,88427.870,171013.866,640236.386,106125.353,9459.079,...,-27.750000,-192961.923667,0.542251,Medium,9,4,1,0,0.816368,0.400234


## 1. Core Factor Distributions

In [2]:
factor_cols = ["flexibility_share", "demand_forecast_error", "net_load"]

fig = make_subplots(rows=1, cols=3, subplot_titles=factor_cols)
for i, col in enumerate(factor_cols, 1):
    fig.add_trace(
        go.Histogram(x=train[col], name=col, nbinsx=80, marker_color="steelblue"),
        row=1, col=i,
    )
fig.update_layout(title="Core Factor Distributions (Train)", showlegend=False, height=350)
fig.show()

## 2. PCA Explained Variance & Biplot

In [10]:
ev = fit.pca.explained_variance_ratio_
fig = go.Figure()
fig.add_trace(go.Bar(x=[f"PC{i+1}" for i in range(3)], y=ev, marker_color="steelblue"))
fig.add_trace(go.Scatter(
    x=[f"PC{i+1}" for i in range(3)], y=np.cumsum(ev),
    mode="lines+markers", name="Cumulative", line=dict(color="firebrick"),
))
fig.update_layout(title="PCA Explained Variance", yaxis_title="Proportion", height=350)
fig.show()

# Biplot: loadings
loadings = fit.pca.components_[:2].T  # (3 factors × 2 PCs)
fig2 = go.Figure()
for i, name in enumerate(factor_cols):
    fig2.add_trace(go.Scatter(
        x=[0, loadings[i, 0]], y=[0, loadings[i, 1]],
        mode="lines+markers+text", text=["", name],
        textposition="top center", name=name,
    ))
fig2.update_layout(title="PCA Biplot (PC1 vs PC2)", xaxis_title="PC1", yaxis_title="PC2",
                   height=400, width=500)
fig2.show()

## 3. Risk Index Time Series with Split Boundaries

In [4]:
# Daily mean for readability
daily = full["risk_index"].resample("D").mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily.index, y=daily.values, mode="lines", name="risk_index (daily mean)",
                         line=dict(width=0.8, color="steelblue")))

for label, ts in [("Train→Val", "2024-01-01"), ("Val→Test", "2024-07-01")]:
    fig.add_vline(x=ts, line_dash="dash", line_color="red")
    fig.add_annotation(x=ts, y=1.05, yref="paper", text=label,
                       showarrow=False, font=dict(color="red"))

fig.update_layout(title="Risk Index Time Series", yaxis_title="risk_index", height=400)
fig.show()

## 4. Risk Index Distribution Overlap Across Splits

In [5]:
fig = go.Figure()
for label, split, color in [("Train", train, "steelblue"), ("Val", val, "orange"), ("Test", test, "green")]:
    fig.add_trace(go.Histogram(
        x=split["risk_index"], name=label, nbinsx=60,
        marker_color=color, opacity=0.5, histnorm="probability density",
    ))
fig.update_layout(title="Risk Index Distribution by Split", barmode="overlay",
                  xaxis_title="risk_index", height=350)
fig.show()

## 5. Risk Category Counts per Split

In [6]:
counts = pd.DataFrame({
    label: split["risk_category"].value_counts()
    for label, split in [("Train", train), ("Val", val), ("Test", test)]
}).T
counts = counts[["Low", "Medium", "High"]]
display(counts)

fig = px.bar(counts.reset_index(), x="index", y=["Low", "Medium", "High"],
             barmode="group", title="Risk Category Counts per Split",
             labels={"index": "Split", "value": "Count"})
fig.update_layout(height=350)
fig.show()

risk_category,Low,Medium,High
Train,8441,8720,8407
Val,2348,1321,648
Test,7238,5035,2525


## 6. Feature Correlation Heatmap vs. Risk Index

In [7]:
from grid_risk.features import FEATURE_NAMES

corr_cols = FEATURE_NAMES + ["risk_index"]
corr = train[corr_cols].corr()

fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r",
                zmin=-1, zmax=1, title="Feature Correlation Matrix (Train)")
fig.update_layout(height=550, width=650)
fig.show()

## 7. Daily-Generation Limitation Check

Within-day variance of `flexibility_share` should be ~0 (daily broadcast), while `net_load` and `demand_forecast_error` should show real hourly variance.

In [8]:
# Compute within-day std for each factor
daily_std = train.groupby(train.index.date)[
    ["flexibility_share", "demand_forecast_error", "net_load"]
].std()

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=["flexibility_share", "demand_forecast_error", "net_load"])
for i, col in enumerate(["flexibility_share", "demand_forecast_error", "net_load"], 1):
    fig.add_trace(
        go.Histogram(x=daily_std[col], nbinsx=60, marker_color="steelblue"),
        row=1, col=i,
    )
fig.update_layout(title="Within-Day Std of Core Factors (Train)",
                  showlegend=False, height=350)
fig.show()

print("Median within-day std:")
print(daily_std.median().to_string())

Median within-day std:
flexibility_share            0.013040
demand_forecast_error      170.950401
net_load                 10927.114220


## Leakage Checks

In [9]:
# Verify scaler mean/std match train-only statistics
train_factors = train[["flexibility_share", "demand_forecast_error", "net_load"]]
print("Scaler mean (fit):", fit.scaler.mean_)
print("Train mean (calc):", train_factors.mean().values)
print()
print("Scaler std  (fit):", fit.scaler.scale_)
print("Train std  (calc):", train_factors.std(ddof=0).values)
print()

# Verify lag alignment: risk_index_lag_24h at t == risk_index at t-24
sample_t = train.index[200]
lag_val = train.loc[sample_t, "risk_index_lag_24h"]
actual_val = full.loc[sample_t - pd.Timedelta(hours=24), "risk_index"]
print(f"Lag check at {sample_t}:")
print(f"  risk_index_lag_24h = {lag_val:.6f}")
print(f"  risk_index[t-24h]  = {actual_val:.6f}")
print(f"  Match: {np.isclose(lag_val, actual_val)}")

Scaler mean (fit): [ 3.48325456e-01  3.53149405e+01 -2.18260996e+05]
Train mean (calc): [ 3.47962204e-01  3.51227120e+01 -2.18607937e+05]

Scaler std  (fit): [1.09196561e-01 2.05775032e+02 8.31276160e+04]
Train std  (calc): [1.09245989e-01 2.05702283e+02 8.31015879e+04]

Lag check at 2021-01-16 07:00:00+00:00:
  risk_index_lag_24h = 0.516505
  risk_index[t-24h]  = 0.516505
  Match: True
